In [ ]:
!git clone https://github.com/d4-5/NLP4.git
%cd NLP4

In [ ]:
!pip install -r requirements.txt

In [ ]:
%cd NLP4

In [1]:
import pandas as pd
import re
from gensim.models import Word2Vec, FastText

from pathlib import Path
import sys

REPO_ROOT = Path('..').resolve()
DATA_DIR = REPO_ROOT / 'data/processed_v2.csv'

In [2]:
def tokenize(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    tokens = re.findall(r'\w+', text)
    return tokens

df = pd.read_csv('../data/processed_v2.csv')
df = df[df['text_clean'].notna()]
df['tokens'] = df['text_clean'].apply(tokenize)
df = df[df['tokens'].map(len) > 2]

sentences = df['tokens'].tolist()
print(f"Документів: {len(sentences)}")
print(f"Приблизно токенів: {sum(len(s) for s in sentences)}")

Документів: 10665
Приблизно токенів: 184123


In [3]:
params = {
    'vector_size': 100,
    'window': 5,
    'min_count': 3,
    'sg': 1,
    'workers': 4,
    'seed': 42
}

w2v_model = Word2Vec(sentences, **params)

ft_model = FastText(sentences, **params)

In [4]:
test_words = ['україни', 'договір', 'тендер', 'грн', 'закупівлі', 'прокуратура', 'корупція', 'суд', 'швидко', 'міськрада']

for word in test_words:
    print(f"\nСлово: {word}")
    if word in w2v_model.wv:
        print(f"  W2V: {[n[0] for n in w2v_model.wv.most_similar(word, topn=5)]}")
    else:
        print("  W2V: OOV")
    print(f"  FT:  {[n[0] for n in ft_model.wv.most_similar(word, topn=5)]}")


Слово: україни
  W2V: ['ради', 'управління', 'обласної', 'щодо', 'державної']
  FT:  ['всеукраїнської', 'українсько', 'українських', 'української', 'українське']

Слово: договір
  W2V: ['оренди', 'уклав', 'рада', 'міськради', 'служби']
  FT:  ['договорі', 'централізованого', 'конкурентного', 'формалізованого', 'договору']

Слово: тендер
  W2V: ['продукції', 'музеїв', 'використання', 'термінології', 'розробку']
  FT:  ['тендери', 'тендері', 'тендеру', 'тендерах', 'уклали']

Слово: грн
  W2V: ['тис', '1', 'суму', '2', '5']
  FT:  ['2', 'суму', 'млн', 'тис', '3']

Слово: закупівлі
  W2V: ['порушення', 'законодавства', 'власності', 'адміністративного', 'межах']
  FT:  ['закупівлю', 'закупівлях', 'закупівля', 'державі', 'державному']

Слово: прокуратура
  W2V: ['вищий', 'розпорядження', 'міністерство', 'дснс', '1991']
  FT:  ['прокуратурі', 'прокуратури', 'генпрокуратура', 'генпрокуратури', 'судом']

Слово: корупція
  W2V: OOV
  FT:  ['кошик', 'корпус', 'корінь', 'фотографіями', 'корпусу']

In [5]:
domain_terms = ['тендер', 'грн', 'договір', 'суд', 'прокуратура']
for word in domain_terms:
    print(f"\nДоменний термін: {word}")
    print(f"  W2V: {[n[0] for n in w2v_model.wv.most_similar(word, topn=5)]}")
    print(f"  FT:  {[n[0] for n in ft_model.wv.most_similar(word, topn=5)]}")


Доменний термін: тендер
  W2V: ['продукції', 'музеїв', 'використання', 'термінології', 'розробку']
  FT:  ['тендери', 'тендері', 'тендеру', 'тендерах', 'уклали']

Доменний термін: грн
  W2V: ['тис', '1', 'суму', '2', '5']
  FT:  ['2', 'суму', 'млн', 'тис', '3']

Доменний термін: договір
  W2V: ['оренди', 'уклав', 'рада', 'міськради', 'служби']
  FT:  ['договорі', 'централізованого', 'конкурентного', 'формалізованого', 'договору']

Доменний термін: суд
  W2V: ['відділення', 'одеської', 'донецької', 'київської', 'визнав']
  FT:  ['суду', 'господарського', 'господарств', 'сільськогосподарського', 'господарської']

Доменний термін: прокуратура
  W2V: ['вищий', 'розпорядження', 'міністерство', 'дснс', '1991']
  FT:  ['прокуратурі', 'прокуратури', 'генпрокуратура', 'генпрокуратури', 'судом']


## 5 Cases: Useful vs Not Useful

### Підсумкова таблиця кейсів

| Word | Type | Useful? | Comment |
| :--- | :--- | :--- | :--- |
| **тендер** | domain | useful | Семантично близькі сусіди: 'торги', 'закупівлі' |
| **грн** | frequent | useful | Близько до інших валют та числових контекстів |
| **міськрада** | domain | useful | FastText краще вловлює морфологічні варіанти |
| **швидко** | general | weak | Сусіди можуть бути близькими стилістично, але не корисні семантично |
| **закупівлі** | morph | useful | Обидві моделі добре працюють з частими термінами, FT краще обробляє суб-слова |

## 7. Word2Vec vs FastText Comparison

- **Word2Vec** добре працює на частих словах з чітким контекстом.
- **FastText** значно кращий для української мови через морфологію (суб-токени) та обробку рідкісних слів.
- Для нашого корпусу (держзакупівлі) FastText виграє на специфічних термінах та їх відмінках.